# Meridian MMM Diagnostics

Use this notebook to fit the model directly and inspect what Meridian is doing — no web app, no Celery, no waiting for the UI.

**Setup:** start the notebook service with:
```bash
docker compose --profile notebook up notebook
```
Then open `http://localhost:8888` in your browser.

In [ ]:
import sys
sys.path.insert(0, '/app')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Path to your CSV — swap to any file under /data/
CSV_PATH = '/data/bike_sales_digital.csv'

df = pd.read_csv(CSV_PATH)
print(f'Rows: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head()

## 1. Data checks

In [ ]:
channels = [c for c in df.columns if c not in ('date', 'acquisitions')]
print('Channels:', channels)

# Zero % per channel
for ch in channels:
    zero_pct = (df[ch] == 0).mean() * 100
    print(f'  {ch}: {zero_pct:.1f}% zeros  |  min={df[ch].min():.0f}  mean={df[ch].mean():.0f}  max={df[ch].max():.0f}')

print(f'\nAcquisitions: min={df.acquisitions.min():.0f}  mean={df.acquisitions.mean():.0f}  max={df.acquisitions.max():.0f}')

# Date spacing
dates = pd.to_datetime(df['date'])
gaps = dates.diff().dropna().dt.days
print(f'\nDate gaps (unique): {sorted(gaps.unique())} — should all be the same')

In [ ]:
# Spend and acquisitions over time
fig, axes = plt.subplots(len(channels) + 1, 1, figsize=(14, 3 * (len(channels) + 1)))
for i, ch in enumerate(channels):
    axes[i].plot(df['date'], df[ch])
    axes[i].set_title(ch)
    axes[i].set_ylabel('Spend')
axes[-1].plot(df['date'], df['acquisitions'], color='green')
axes[-1].set_title('acquisitions')
plt.tight_layout()
plt.show()

In [ ]:
# Channel spend correlations — high correlation = hard to separate effects
print('Spend correlations:')
df[channels].corr().round(2)

## 2. Normalisation check
This is what the model actually receives — verify values are in a sensible range.

In [ ]:
media_scale = {ch: float(df[ch].max()) for ch in channels}
kpi_scale = float(df['acquisitions'].mean())

print('media_scale (max spend per channel):')
for ch, s in media_scale.items():
    print(f'  {ch}: {s:.0f}')

print(f'\nkpi_scale (mean acquisitions): {kpi_scale:.1f}')

# Normalised media: should be in [0, 1]
print('\nNormalised media stats (should be in [0, 1]):')
for ch in channels:
    normed = df[ch] / media_scale[ch]
    print(f'  {ch}: min={normed.min():.3f}  mean={normed.mean():.3f}  max={normed.max():.3f}')

# Normalised KPI: should be ~1
normed_kpi = df['acquisitions'] / kpi_scale
print(f'\nNormalised KPI: min={normed_kpi.min():.3f}  mean={normed_kpi.mean():.3f}  max={normed_kpi.max():.3f}')

## 3. Fit the model
Runs MCMC directly — adjust `n_chains`, `n_warmup`, `n_samples` below.

In [ ]:
from app.mmm.meridian_wrapper import MeridianWrapper
from app.models.run import MeridianConfig

config = MeridianConfig(
    n_chains=2,
    n_warmup=500,
    n_samples=200,
)

wrapper = MeridianWrapper(config)
fit_result = wrapper.fit(df=df, channel_names=channels, granularity='weekly')
print(f'\nDone. r_hat_max={fit_result.r_hat_max:.3f}  ess_bulk_min={fit_result.ess_bulk_min}')

## 4. Inspect posterior samples
Look at the actual parameter distributions Meridian estimated.

In [ ]:
posterior = fit_result.posterior

print('Posterior parameter means per channel:')
print(f'  {"channel":30s}  {"alpha":>8}  {"ec":>8}  {"slope":>8}  {"beta":>8}')
for i, ch in enumerate(channels):
    a = posterior.alpha[:, i].mean()
    e = posterior.ec[:, i].mean()
    s = posterior.slope[:, i].mean()
    b = posterior.beta[:, i].mean()
    print(f'  {ch:30s}  {a:8.3f}  {e:8.3f}  {s:8.3f}  {b:8.3f}')

print()
print('Notes:')
print('  alpha: adstock decay [0-1). High = long carryover. ~0 = no memory.')
print('  ec:    half-saturation point in NORMALISED spend [0-1]. ~0.5 = saturates at 50% of max.')
print('  slope: Hill curve steepness. >1 = S-curve. <1 = concave.')
print('  beta:  channel scaling coefficient (in normalised KPI units).')

In [ ]:
# Distribution of EC across posterior samples — narrow = well identified, wide = uncertain
fig, axes = plt.subplots(1, len(channels), figsize=(5 * len(channels), 3))
if len(channels) == 1:
    axes = [axes]
for i, ch in enumerate(channels):
    axes[i].hist(posterior.ec[:, i], bins=40, color='steelblue', edgecolor='white')
    axes[i].set_title(f'{ch}\nEC distribution')
    axes[i].set_xlabel('EC (normalised)')
plt.tight_layout()
plt.show()

In [ ]:
# Trace plots — chains should overlap and mix well
# If chains are separated = poor convergence = high R-hat
mmm = fit_result.mmm
posterior_raw = mmm.inference_data.posterior

var = 'ec_m'  # change to alpha_m, slope_m, beta_gm as needed
data = posterior_raw[var].values  # (chains, draws, channels)

n_chains, n_draws = data.shape[0], data.shape[1]
fig, axes = plt.subplots(1, len(channels), figsize=(5 * len(channels), 3))
if len(channels) == 1:
    axes = [axes]
for i, ch in enumerate(channels):
    for c in range(n_chains):
        axes[i].plot(data[c, :, i], alpha=0.7, label=f'chain {c+1}')
    axes[i].set_title(f'{ch}\n{var} trace')
    axes[i].legend(fontsize=7)
plt.suptitle('Trace plots — chains should overlap (good mixing)')
plt.tight_layout()
plt.show()

In [ ]:
# Full arviz summary
import arviz as az
summary = az.summary(mmm.inference_data, var_names=['alpha_m', 'ec_m', 'slope_m', 'beta_gm'])
print(summary.to_string())

## 5. Response curves

In [ ]:
from app.mmm.response_curves import extract_response_curves

curves = extract_response_curves(fit_result)

fig, axes = plt.subplots(1, len(channels), figsize=(5 * len(channels), 4))
if len(channels) == 1:
    axes = [axes]
for i, ch in enumerate(channels):
    c = curves[ch]
    axes[i].plot(c.spend_points, c.acquisitions, label='mean', color='steelblue')
    axes[i].fill_between(c.spend_points, c.ci_lower, c.ci_upper, alpha=0.2, color='steelblue', label='80% CI')
    axes[i].set_title(ch)
    axes[i].set_xlabel('Spend ($)')
    axes[i].set_ylabel('Acquisitions')
    axes[i].legend()
plt.suptitle('Response curves — should be concave (diminishing returns)')
plt.tight_layout()
plt.show()

# Print curve value ranges
for ch, c in curves.items():
    print(f'{ch}: acq at spend=0: {c.acquisitions[0]:.1f}  at max spend: {c.acquisitions[-1]:.1f}')

## 6. Budget optimisation

In [ ]:
from app.mmm.budget_optimizer import optimize_budget, compute_total_acquisitions, compute_prior_allocation

total_budget = df[channels].sum(axis=1).mean()  # average historical weekly budget
print(f'Total budget (avg weekly): ${total_budget:,.0f}')

prior = compute_prior_allocation(df, channels, total_budget)
optimal = optimize_budget(curves, total_budget)

prior_acq = compute_total_acquisitions(curves, prior)
optimal_acq = compute_total_acquisitions(curves, optimal)
lift = (optimal_acq - prior_acq) / prior_acq * 100

print(f'\n{"Channel":30s}  {"Prior":>12}  {"Optimal":>12}  {"Change":>10}')
for ch in channels:
    delta = optimal[ch] - prior[ch]
    print(f'{ch:30s}  ${prior[ch]:>10,.0f}  ${optimal[ch]:>10,.0f}  {delta:>+10,.0f}')
print(f'\nPrior acquisitions:   {prior_acq:.1f}')
print(f'Optimal acquisitions: {optimal_acq:.1f}')
print(f'Lift: {lift:+.1f}%')